# Структура
00 - init  
01 - Склейка данных: станции + координаты + погода + аварии <br>
02 - EDA по таблице Cell_Avail  
03 - EDA по таблице EDA по таблице main_data_with_weather  
04 - Baseline (LogReg, RF, NB)  
05 - Кластеризация (попытка)  
06 - Признаки  
07 - Бустинги  
08 - Нейросети  
09 - Кросс-валидация  

Папка: https://drive.google.com/drive/folders/1xeN-a_W6E6YVM2I06JZf_NRde3JGpyrV?usp=sharing
#### Файлы:
*   main_data_with_weather.xlsx
*   Cell_Avail.xlsx
*   северо-запад_Rollout_RF-Site_vsf_20260224.xlsx
*   NOC_TT_Report_23.xlsx
*   NOC_TT_Report_24.xlsx
*   NOC_TT_Report_25.xlsx


### 00: init

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from sklearn.metrics import classification_report, recall_score, roc_auc_score, confusion_matrix, accuracy_score, precision_score, f1_score, average_precision_score

In [ ]:
df_main_with_weather = pd.read_excel("main_data_with_weather.xlsx")

tech_cols = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']

tech_cols = [col for col in tech_cols if col in df_main_with_weather.columns]
weather_cols = [col for col in weather_cols if col in df_main_with_weather.columns]

In [ ]:
df_main_with_weather.columns.tolist()

In [ ]:
df_cell_avail = pd.read_excel("Cell_Avail.xlsx")

In [ ]:
df2 = pd.read_excel("северо-запад_Rollout_RF-Site_vsf_20260224.xlsx")

In [ ]:
df_alarms = pd.read_excel('NOC_TT_Report_23.xlsx')

### 01: Склейка данных

Склейка `Cell_Avail`, `северо-запад_Rollout_RF-Site_vsf_20260224`, `NOC_TT_Report_23` (24 и 25 локально).  
Результат: `availability_with_alarms_merged` с колонками:  
RECDATE, Master Site, Region, Subregion,  
Cell Avail 2G (%), Cell Avail 3G (%), Cell Avail 4G (%), Cell Avail Base Tech (%),  
Номер сайта, Субрегион, Тип объекта, Адрес, Производитель БС,  
Широта WGS84, Долгота WGS84, Дата запуска, Высота подвеса, м,  
Alarm Count, Alarm Descriptions, Total Fault Time, Service

In [ ]:
df_temp = df.copy()
df_temp['Master Site'] = df_temp['Master Site'].str.replace(r'([A-Za-z]+)00', r'\1', regex=True)

result = pd.merge(df_temp, df2, left_on='Master Site', right_on='Номер сайта', how='left')
result.head()

In [ ]:
df_avail = result

df_alarms['Alarm Up'] = pd.to_datetime(df_alarms['Alarm Up'], format='%m/%d/%y %H:%M', errors='coerce')
df_alarms['Closed Time'] = pd.to_datetime(df_alarms['Closed Time'], format='%m/%d/%y %H:%M', errors='coerce')
df_avail['RECDATE'] = pd.to_datetime(df_avail['RECDATE'])
df_avail['RECDATE_DATE'] = df_avail['RECDATE'].dt.date

nat_count = df_alarms['Alarm Up'].isna().sum()

df_alarms_clean = df_alarms.dropna(subset=['Alarm Up', 'Closed Time']).copy()

df_alarms_clean['Alarm description'] = df_alarms_clean['Alarm description'].fillna('Unknown').astype(str)
df_alarms_clean['Entry ID'] = df_alarms_clean['Entry ID'].fillna('Unknown').astype(str)
df_alarms_clean['Service'] = df_alarms_clean['Service'].fillna('Unknown').astype(str)
df_alarms_clean['Fault-time per TT'] = pd.to_numeric(df_alarms_clean['Fault-time per TT'], errors='coerce').fillna(0)

def get_affected_dates(alarm_up, closed_time):
    dates = []
    current_date = alarm_up.date()
    end_date = closed_time.date()

    while current_date <= end_date:
        dates.append(current_date)
        current_date += timedelta(days=1)

    return dates

expanded_alarms = []

for idx, row in df_alarms_clean.iterrows():
    affected_dates = get_affected_dates(row['Alarm Up'], row['Closed Time'])

    for date in affected_dates:
        alarm_copy = row.copy()
        alarm_copy['Affected Date'] = date
        expanded_alarms.append(alarm_copy)

df_expanded = pd.DataFrame(expanded_alarms)

alarm_summary = df_expanded.groupby(['Hostname', 'Affected Date']).agg({
    'Entry ID': 'count',
    'Alarm description': lambda x: ' | '.join(x.unique()),
    'Fault-time per TT': 'sum',
    'Service': 'first',
}).reset_index()

alarm_summary.columns = ['Master Site', 'RECDATE_DATE', 'Alarm Count', 'Alarm Descriptions', 'Total Fault Time', 'Service']


df_result = df_avail.merge(
    alarm_summary,
    left_on=['Master Site', 'RECDATE_DATE'],
    right_on=['Master Site', 'RECDATE_DATE'],
    how='left'
)

df_result['Alarm Count'] = df_result['Alarm Count'].fillna(0).astype(int)
df_result['Alarm Descriptions'] = df_result['Alarm Descriptions'].fillna('No alarms')
df_result['Total Fault Time'] = df_result['Total Fault Time'].fillna(0)
df_result['Service'] = df_result['Service'].fillna('Unknown')

df_final = df_result.drop(columns=['RECDATE_DATE'])
df_final.to_excel('availability_with_alarms_merged.xlsx', index=False)

In [ ]:
df_final.head()

### 02: EDA по таблице Cell_Avail (доступности)


In [ ]:
df = df_cell_avail.copy()

In [ ]:
# Преобразование дат
df['RECDATE'] = pd.to_datetime(df['RECDATE'])

# Добавление временных признаков
df['Year'] = df['RECDATE'].dt.year
df['Month'] = df['RECDATE'].dt.month
df['DayOfWeek'] = df['RECDATE'].dt.dayofweek  # 0=Monday, 6=Sunday
df['DayName'] = df['RECDATE'].dt.day_name()
df['MonthName'] = df['RECDATE'].dt.month_name()

# Колонки с доступностью
avail_cols = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
for col in avail_cols:
  df[col] = 100 - df[col]
print(f"Всего записей: {len(df)}")
print(f"Период: {df['RECDATE'].min()} - {df['RECDATE'].max()}")
print(f"Уникальных станций: {df['Master Site'].nunique()}")

- Средняя доступность по дням
- Средняя доступность по технологиям
- Средняя доступность по годам
- Распределение по годам  
Сохранено в `1_time_series.png`
####

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Среднее по всем станциям по времени
daily_mean = df.groupby('RECDATE')['Cell Avail Base Tech (%)'].mean().reset_index()

for year in sorted(df['Year'].unique()):
    year_data = daily_mean[daily_mean['RECDATE'].dt.year == year]
    axes[0, 0].plot(year_data['RECDATE'], year_data['Cell Avail Base Tech (%)'],
                    label=str(year), linewidth=1.5, alpha=0.8)

axes[0, 0].set_xlabel('Дата', fontsize=12)
axes[0, 0].set_ylabel('Cell Avail Base Tech (%)', fontsize=12)
axes[0, 0].set_title('Средняя доступность по дням (по годам)', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Отдельно по каждой технологии
for col in avail_cols[:3]:  # 2G, 3G, 4G
    col_mean = df.groupby('RECDATE')[col].mean()
    axes[0, 1].plot(col_mean.index, col_mean.values, label=col, linewidth=1.5, alpha=0.8)

axes[0, 1].set_xlabel('Дата', fontsize=12)
axes[0, 1].set_ylabel('Availability (%)', fontsize=12)
axes[0, 1].set_title('Средняя доступность по технологиям', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Распределение по годам
yearly_stats = df.groupby('Year')['Cell Avail Base Tech (%)'].agg(['mean', 'median', 'std']).reset_index()
axes[1, 0].bar(yearly_stats['Year'].astype(str), yearly_stats['mean'],
               yerr=yearly_stats['std'], capsize=5, alpha=0.7, color='skyblue')
axes[1, 0].set_xlabel('Год', fontsize=12)
axes[1, 0].set_ylabel('Средняя доступность (%)', fontsize=12)
axes[1, 0].set_title('Средняя доступность по годам', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Boxplot по годам
df.boxplot(column='Cell Avail Base Tech (%)', by='Year', ax=axes[1, 1])
axes[1, 1].set_xlabel('Год', fontsize=12)
axes[1, 1].set_ylabel('Cell Avail Base Tech (%)', fontsize=12)
axes[1, 1].set_title('Распределение по годам', fontsize=14, fontweight='bold')
plt.suptitle('')  # убираем автоматический заголовок

plt.tight_layout()
plt.savefig('1_time_series.png', dpi=300, bbox_inches='tight')
plt.show()

- Средняя доступность по месяцам  
- Средняя доступность по дням недели  
- Heatmap: месяцы х дни недели  
Сохранено в `2_monthly_weekly.png`
####

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# По месяцам (среднее)
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_mean = df.groupby('MonthName')['Cell Avail Base Tech (%)'].mean().reindex(month_order)

axes[0, 0].bar(monthly_mean.index, monthly_mean.values, color='coral', alpha=0.7)
axes[0, 0].set_xlabel('Месяц', fontsize=12)
axes[0, 0].set_ylabel('Средняя доступность (%)', fontsize=12)
axes[0, 0].set_title('Средняя доступность по месяцам', fontsize=14, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# По дням недели
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_mean = df.groupby('DayName')['Cell Avail Base Tech (%)'].mean().reindex(day_order)

axes[0, 1].bar(daily_mean.index, daily_mean.values, color='green', alpha=0.7)
axes[0, 1].set_xlabel('День недели', fontsize=12)
axes[0, 1].set_ylabel('Средняя доступность (%)', fontsize=12)
axes[0, 1].set_title('Средняя доступность по дням недели', fontsize=14, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Heatmap: месяцы × дни недели
pivot_data = df.groupby(['MonthName', 'DayName'])['Cell Avail Base Tech (%)'].mean().unstack()
pivot_data = pivot_data.reindex(index=month_order, columns=day_order)

sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1, 0],
            cbar_kws={'label': 'Средняя доступность (%)'})
axes[1, 0].set_title('Тепловая карта: Месяцы × Дни недели', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('2_monthly_weekly.png', dpi=300, bbox_inches='tight')
plt.show()

- Pairplot для всех колонок доступности  
Сохранено в `3_scatter_pairs.png`
####

In [ ]:
# Pairplot для всех колонок доступности
pair_df = df[avail_cols].dropna()

g = sns.pairplot(pair_df, plot_kws={'alpha': 0.3, 's': 20},
                 diag_kind='kde', corner=True)
g.fig.suptitle('Pairplot: Зависимости между технологиями', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('3_scatter_pairs.png', dpi=300, bbox_inches='tight')
plt.show()

- Попарные scatter'ы технологий (2G vs 3G, 2G vs 4G, etc)  
Сохранено в `3_scatter_detailed.png`
####

In [ ]:
# Отдельные scatter для 2G vs 3G, 3G vs 4G, etc.
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

pairs = [('Cell Avail 2G (%)', 'Cell Avail 3G (%)'),
         ('Cell Avail 3G (%)', 'Cell Avail 4G (%)'),
         ('Cell Avail 2G (%)', 'Cell Avail 4G (%)'),
         ('Cell Avail 4G (%)', 'Cell Avail Base Tech (%)')]

for idx, (x_col, y_col) in enumerate(pairs):
    scatter_data = df[[x_col, y_col]].dropna()
    axes[idx].scatter(scatter_data[x_col], scatter_data[y_col],
                     alpha=0.3, s=20, color='blue')
    axes[idx].set_xlabel(x_col, fontsize=11)
    axes[idx].set_ylabel(y_col, fontsize=11)
    axes[idx].set_title(f'{x_col} vs {y_col}', fontsize=12, fontweight='bold')
    axes[idx].plot([0, 20], [0, 20], 'r--', alpha=0.5, label='y=x')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('3_scatter_detailed.png', dpi=300, bbox_inches='tight')
plt.show()


- Записи, где хотя бы одна <100%
- Матрица корреляций
- Heatmap корреляций
- Корреляции с Base Tech  
Сохранено в `4_correlation_matrix.png`
####

In [ ]:
# Фильтр: хотя бы одна характеристика < 100%
mask_less_100 = (df[avail_cols] > 0).any(axis=1)
df_less_100 = df[mask_less_100][avail_cols]

print(f"Записей где хотя бы одна < 100%: {len(df_less_100)} ({len(df_less_100)/len(df)*100:.2f}%)")

# Матрица корреляций
corr_matrix = df_less_100.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap корреляций
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=1, ax=axes[0],
            cbar_kws={'label': 'Коэффициент корреляции'})
axes[0].set_title('Матрица корреляций (когда < 100%)', fontsize=14, fontweight='bold')

# Корреляции с Base Tech
base_corr = corr_matrix['Cell Avail Base Tech (%)'].drop('Cell Avail Base Tech (%)')
colors = ['red' if x < 0 else 'green' for x in base_corr.values]
bars = axes[1].bar(range(len(base_corr)), base_corr.values, color=colors, alpha=0.7)
axes[1].set_xticks(range(len(base_corr)))
axes[1].set_xticklabels(base_corr.index, rotation=45, ha='right')
axes[1].set_ylabel('Коэффициент корреляции', fontsize=12)
axes[1].set_title('Корреляция с Cell Avail Base Tech', fontsize=14, fontweight='bold')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('4_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nМатрица корреляций (когда хотя бы одна < 100%):")
print(corr_matrix.round(3))



- Распределение доступности Base Tech
- Распределение значений <100%
- Распределение значений <98%
- Соотношение записей (100% vs <100%)  
Сохранено в `base_tech_distribution.png`
####

In [ ]:
df = df_cell_avail.copy()
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# 1. Основная гистограмма
axes[0, 0].hist(df['Cell Avail Base Tech (%)'].dropna(),
                bins=50,
                edgecolor='black',
                alpha=0.7,
                color='#0F2D69')  # R15 G45 B105
axes[0, 0].set_xlabel('Cell Avail Base Tech (%)', fontsize=14)
axes[0, 0].set_ylabel('Количество записей', fontsize=14)
axes[0, 0].set_title('Распределение доступности Base Tech', fontsize=16, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')
axes[0, 0].tick_params(labelsize=12)

# 2. Гистограмма с фокусом на значения < 100%
df_problems = df[df['Cell Avail Base Tech (%)'] < 100]
if len(df_problems) > 0:
    axes[0, 1].hist(df_problems['Cell Avail Base Tech (%)'].dropna(),
                    bins=30,
                    edgecolor='black',
                    alpha=0.7,
                    color='#0FA0D7')  # R15 G160 B215
    axes[0, 1].set_xlabel('Cell Avail Base Tech (%)', fontsize=14)
    axes[0, 1].set_ylabel('Количество записей', fontsize=14)
    axes[0, 1].set_title('Распределение проблем (< 100%)', fontsize=16, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    axes[0, 1].tick_params(labelsize=12)
else:
    axes[0, 1].text(0.5, 0.5, 'Нет записей с доступностью < 100%',
                    ha='center', va='center', fontsize=14, transform=axes[0, 1].transAxes)
    axes[0, 1].set_xlim(0, 1)
    axes[0, 1].set_ylim(0, 1)
#3. Гистограмма с фокусом на значения < 98%
df_problems = df[df['Cell Avail Base Tech (%)'] < 98]
if len(df_problems) > 0:
    axes[1, 1].hist(df_problems['Cell Avail Base Tech (%)'].dropna(),
                    bins=50,
                    edgecolor='black',
                    alpha=0.7,
                    color='#0FA0D7')  # R15 G160 B215
    axes[1, 1].set_xlabel('Cell Avail Base Tech (%)', fontsize=14)
    axes[1, 1].set_ylabel('Количество записей', fontsize=14)
    axes[1, 1].set_title('Распределение проблем (< 98%)', fontsize=16, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].tick_params(labelsize=12)
else:
    axes[1, 1].text(0.5, 0.5, 'Нет записей с доступностью < 100%',
                    ha='center', va='center', fontsize=14, transform=axes[0, 1].transAxes)
    axes[1, 1].set_xlim(0, 1)
    axes[1, 1].set_ylim(0, 1)

# 4. Pie chart: 100% vs < 100%
count_100 = (df['Cell Avail Base Tech (%)'] == 100).sum()
count_less_100 = (df['Cell Avail Base Tech (%)'] < 100).sum()
total = len(df)


axes[1, 0].pie([count_100, count_less_100],
               labels=['100%', '< 100%'],
               autopct='%1.1f%%',
               colors=['#7DA0D2', '#CDDCF0'],  # R125 G160 B210 и R205 G220 B240
               explode=[0, 0.1],
               textprops={'fontsize': 14})
axes[1, 0].set_title(f'Соотношение записей\n(всего: {total})', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('base_tech_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

- Распределение >=98%
####

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(14, 10))
df_problems = df[(df['Cell Avail Base Tech (%)'] >= 98) & (df['Cell Avail Base Tech (%)'] < 100)]
if len(df_problems) > 0:
    axes.hist(df_problems['Cell Avail Base Tech (%)'].dropna(),
                    bins=100,
                    edgecolor='black',
                    alpha=0.7,
                    color='salmon')
    axes.set_xlabel('Cell Avail Base Tech (%)', fontsize=12)
    axes.set_ylabel('Количество записей', fontsize=12)
    axes.set_title('Распределение проблем (>= 98%)', fontsize=14, fontweight='bold')
    axes.grid(True, alpha=0.3, axis='y')
else:
    axes.text(0.5, 0.5, 'Нет записей с доступностью < 100%',
                    ha='center', va='center', fontsize=12, transform=axes[0, 1].transAxes)
    axes.set_xlim(0, 1)
    axes.set_ylim(0, 1)

- Гистограмма отраженных данных (100 - Avail)
- QQ-plot отраженных данных
####

In [ ]:
df_filtered = df[(df['Cell Avail Base Tech (%)'] < 100) & (df['Cell Avail Base Tech (%)'] > 0)]
data = df_filtered['Cell Avail Base Tech (%)'].dropna()

data_reflected = 100 - data

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(data_reflected, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0].set_title('Гистограмма (100 - Avail)')
axes[0].set_xlabel('Расстояние до 100%')

rate_param = 1 / data_reflected.mean()
exp_quantiles = stats.expon.ppf(np.linspace(0.01, 0.99, len(data_reflected)), scale=1/rate_param)
data_sorted = np.sort(data_reflected)

axes[1].scatter(exp_quantiles, data_sorted, alpha=0.5, s=10, color='purple')
axes[1].plot([min(exp_quantiles), max(exp_quantiles)],
             [min(exp_quantiles), max(exp_quantiles)], 'r--')
axes[1].set_title('QQ-plot для отраженных данных')

plt.tight_layout()
plt.show()

ks_stat, ks_pvalue = stats.kstest(data_reflected, 'expon', args=(0, 1/rate_param))
print(f"KS-тест для отраженных данных: p-value = {ks_pvalue:.4f}")
if ks_pvalue > 0.05:
    print("Гипотеза об экспоненциальности (для 100-X) подтверждена")

- Топ 10 вышек по с худшей доступностью
####

In [ ]:
avg_availability_by_tower = df.groupby('Master Site')['Cell Avail Base Tech (%)'].agg([
    'mean',      # среднее значение доступности
    'median',    # медиана
    'std',       # стандартное отклонение
    'count',     # количество записей (дней)
    'min',       # минимум
    'max'        # максимум
]).round(2)

avg_availability_by_tower.columns = ['Средняя доступность (%)',
                                      'Медиана (%)',
                                      'Стд. отклонение',
                                      'Кол-во дней',
                                      'Мин (%)',
                                      'Макс (%)']

print("Статистика по вышкам:")
print(avg_availability_by_tower.sort_values('Средняя доступность (%)', ascending=False))


if 'Date' in df.columns:
    stats_with_dates = df.groupby('Cell ID').agg({
        'Cell Avail Base Tech (%)': 'mean',
        'Date': 'nunique'  # количество уникальных дней
    }).round(2)

    stats_with_dates.columns = ['Средняя доступность (%)', 'Уникальных дней']
    print("\nСтатистика с уникальными днями:")
    print(stats_with_dates)

top_10_worst = avg_availability_by_tower.nsmallest(10, 'Средняя доступность (%)')

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_10_worst)), top_10_worst['Средняя доступность (%)'])
plt.yticks(range(len(top_10_worst)), top_10_worst.index)
plt.xlabel('Средняя доступность (%)')
plt.title('Топ-10 вышек с худшей доступностью')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

avg_availability_by_tower.to_csv('tower_availability_stats.csv')

Random Forest, Gradient Boosting
####

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [ ]:
df = df_cell_avail.copy()
df['Target_Next_Day'] = df.groupby('Master Site')['Cell Avail Base Tech (%)'].shift(-1)
df_model = df.dropna(subset=['Target_Next_Day'])

bins = [0, 50, 70, 80, 90, 95, 98, 99, 100]
labels = ['0-50%', '50-70%', '70-80%', '80-90%', '90-95%', '95-98%', '98-99%', '99-100%']
df_model['availability_class'] = pd.cut(df_model['Target_Next_Day'], bins=bins, labels=labels, include_lowest=True)

numeric_features = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
categorical_features = ['Master Site', 'Region', 'Subregion']

df_encoded = pd.get_dummies(df_model, columns=categorical_features, drop_first=True)

feature_cols = numeric_features + [col for col in df_encoded.columns if any(cat in col for cat in categorical_features)]

X = df_encoded[feature_cols].fillna(0)
y = df_encoded['availability_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=200, class_weight='balanced', min_samples_split=2, min_samples_leaf=1, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("RandomForest Classification Report:")
print(classification_report(y_test, y_pred))

print("Recall by Class:")
recall_per_class = recall_score(y_test, y_pred, average=None, labels=labels)
for cls, rec in zip(labels, recall_per_class):
    print(f"{cls}: Recall = {rec:.4f}")

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print("\nGradientBoosting Classification Report:")
print(classification_report(y_test, y_pred_gb))

### 03: EDA по таблице main_data_with_weather (погоды)

In [ ]:
df = df_main_with_weather.copy()

tech_cols = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']

tech_cols = [col for col in tech_cols if col in df.columns]
weather_cols = [col for col in weather_cols if col in df.columns]

- Корреляция (все данные)
- Корреляция (доступность <99.8%)
- Влияние погоды на Cell Avail Base Tech (%) (все данные)
- Влияние погоды на Cell Avail Base Tech (%) (доступность <99.8%)  
Сохранено в `correlation_analysis.png`
- Корреляция между сгенерированными признаками
- Корреляция сгенерированных признаков с таргетом
####

In [ ]:
df_analysis = df[tech_cols + weather_cols].copy()
for col in df_analysis.columns:
    df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')
df_analysis.dropna(inplace=True)

corr_matrix = df_analysis.corr()

df_problems = df_analysis[df_analysis['Cell Avail Base Tech (%)'] < 99.8].copy()
corr_problems = df_problems[tech_cols + weather_cols].corr() if len(df_problems) > 0 else None

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if len(tech_cols) > 0 and len(weather_cols) > 0:
    corr_subset = corr_matrix.loc[tech_cols, weather_cols]
    sns.heatmap(corr_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 0],
                annot_kws={'size': 12})
    axes[0, 0].set_title('Корреляция: Все данные', fontsize=16, fontweight='bold')
    axes[0, 0].tick_params(labelsize=12)
    plt.setp(axes[0, 0].get_xticklabels(), rotation=45, ha='right', fontsize=12)
    plt.setp(axes[0, 0].get_yticklabels(), fontsize=12)

if corr_problems is not None and len(df_problems) > 0:
    corr_problems_subset = corr_problems.loc[tech_cols, weather_cols]
    sns.heatmap(corr_problems_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 1],
                annot_kws={'size': 12})
    axes[0, 1].set_title(f'Корреляция: Доступность < 99.8% (n={len(df_problems)})', fontsize=16, fontweight='bold')
    axes[0, 1].tick_params(labelsize=12)
    plt.setp(axes[0, 1].get_xticklabels(), rotation=45, ha='right', fontsize=12)
    plt.setp(axes[0, 1].get_yticklabels(), fontsize=12)

fail_col = 'Cell Avail Base Tech (%)' if 'Cell Avail Base Tech (%)' in tech_cols else tech_cols[0] if tech_cols else None
if fail_col and len(weather_cols) > 0:
    fail_corrs = corr_matrix[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs.values]  # R15 G45 B105 и R15 G160 B215
    axes[1, 0].barh(range(len(fail_corrs)), fail_corrs.values, color=colors, alpha=0.8)
    axes[1, 0].set_yticks(range(len(fail_corrs)))
    axes[1, 0].set_yticklabels(fail_corrs.index, fontsize=12)
    axes[1, 0].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 0].set_xlabel('Коэффициент корреляции', fontsize=14)
    axes[1, 0].set_title(f'Влияние погоды на {fail_col} (все данные)', fontsize=16, fontweight='bold')
    axes[1, 0].set_xlim([-1, 1])
    axes[1, 0].tick_params(labelsize=12)
    for i, v in enumerate(fail_corrs.values):
        axes[1, 0].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=11)

if fail_col and len(weather_cols) > 0 and corr_problems is not None:
    fail_corrs_problems = corr_problems[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs_problems.values]  # R15 G45 B105 и R15 G160 B215
    axes[1, 1].barh(range(len(fail_corrs_problems)), fail_corrs_problems.values, color=colors, alpha=0.8)
    axes[1, 1].set_yticks(range(len(fail_corrs_problems)))
    axes[1, 1].set_yticklabels(fail_corrs_problems.index, fontsize=12)
    axes[1, 1].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 1].set_xlabel('Коэффициент корреляции', fontsize=14)
    axes[1, 1].set_title(f'Влияние погоды на {fail_col} (< 99.8%)', fontsize=16, fontweight='bold')
    axes[1, 1].set_xlim([-1, 1])
    axes[1, 1].tick_params(labelsize=12)
    for i, v in enumerate(fail_corrs_problems.values):
        axes[1, 1].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=11)

plt.suptitle('Анализ корреляции', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nВСЕГО ЗАПИСЕЙ: {len(df_analysis)}")
print(f"ЗАПИСЕЙ С ДОСТУПНОСТЬЮ < 99.8%: {len(df_problems)}\n")

if fail_col and len(weather_cols) > 0:
    print(f"КОРРЕЛЯЦИИ С {fail_col} (ВСЕ ДАННЫЕ):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = df_analysis[fail_col].corr(df_analysis[weather_col])
        p_value = stats.pearsonr(df_analysis[fail_col].dropna(), df_analysis[weather_col].dropna())[1]
        print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

if corr_problems is not None and len(df_problems) > 0:
    print(f"\nКОРРЕЛЯЦИИ С {fail_col} (ДОСТУПНОСТЬ < 99.8%):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = corr_problems[fail_col][weather_col]
        if len(df_problems[weather_col].dropna()) > 2:
            p_value = stats.pearsonr(df_problems[fail_col].dropna(), df_problems[weather_col].dropna())[1]
            print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

- Корреляция без сдвига (все данные)
- Корреляция (Погода со сдвигом +1 день)
- Влияние погоды на Cell Avail Base Tech (%) (без сдвига)
- Влияние погоды на Cell Avail Base Tech (%) (сдвиг +1 день)  
Сохранено в `correlation_with_shift.png`
####

In [ ]:
if 'Date' in df.columns or 'Дата' in df.columns:
    date_col = 'Date' if 'Date' in df.columns else 'Дата'
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    df_analysis = df[tech_cols + weather_cols].copy()
    for col in df_analysis.columns:
        df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')
    df_analysis.dropna(inplace=True)

    df_shifted = df[tech_cols + weather_cols + [date_col]].copy()
    for col in weather_cols:
        df_shifted[col] = df_shifted[col].shift(1)
    for col in df_shifted.columns:
        if col != date_col:
            df_shifted[col] = pd.to_numeric(df_shifted[col], errors='coerce')
    df_shifted.dropna(inplace=True)

    corr_matrix = df_analysis.corr()
    corr_shifted = df_shifted[tech_cols + weather_cols].corr() if len(df_shifted) > 0 else None
else:
    df_analysis = df[tech_cols + weather_cols].copy()
    for col in df_analysis.columns:
        df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')
    df_analysis.dropna(inplace=True)
    corr_matrix = df_analysis.corr()
    corr_shifted = None
    print("ВНИМАНИЕ: Колонка с датой не найдена. Анализ со сдвигом невозможен.")

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

if len(tech_cols) > 0 and len(weather_cols) > 0:
    corr_subset = corr_matrix.loc[tech_cols, weather_cols]
    sns.heatmap(corr_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 0],
                annot_kws={'size': 11})
    axes[0, 0].set_title('Корреляция: Все данные (без сдвига)', fontsize=14, fontweight='bold')
    axes[0, 0].tick_params(labelsize=10)
    plt.setp(axes[0, 0].get_xticklabels(), rotation=45, ha='right', fontsize=10)
    plt.setp(axes[0, 0].get_yticklabels(), fontsize=10)

if corr_shifted is not None and len(df_shifted) > 0:
    corr_shifted_subset = corr_shifted.loc[tech_cols, weather_cols]
    sns.heatmap(corr_shifted_subset, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=axes[0, 1],
                annot_kws={'size': 11})
    axes[0, 1].set_title(f'Корреляция: Погода со сдвигом +1 день (n={len(df_shifted)})', fontsize=14, fontweight='bold')
    axes[0, 1].tick_params(labelsize=10)
    plt.setp(axes[0, 1].get_xticklabels(), rotation=45, ha='right', fontsize=10)
    plt.setp(axes[0, 1].get_yticklabels(), fontsize=10)

fail_col = 'Cell Avail Base Tech (%)' if 'Cell Avail Base Tech (%)' in tech_cols else tech_cols[0] if tech_cols else None
if fail_col and len(weather_cols) > 0:
    fail_corrs = corr_matrix[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs.values]
    axes[1, 0].barh(range(len(fail_corrs)), fail_corrs.values, color=colors, alpha=0.8)
    axes[1, 0].set_yticks(range(len(fail_corrs)))
    axes[1, 0].set_yticklabels(fail_corrs.index, fontsize=11)
    axes[1, 0].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 0].set_xlabel('Коэффициент корреляции', fontsize=12)
    axes[1, 0].set_title(f'Влияние погоды на {fail_col} (без сдвига)', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlim([-1, 1])
    axes[1, 0].tick_params(labelsize=10)
    for i, v in enumerate(fail_corrs.values):
        axes[1, 0].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=10)

if fail_col and len(weather_cols) > 0 and corr_shifted is not None:
    fail_corrs_shifted = corr_shifted[fail_col][weather_cols].sort_values(ascending=False)
    colors = ['#0F2D69' if x > 0 else '#0FA0D7' for x in fail_corrs_shifted.values]
    axes[1, 1].barh(range(len(fail_corrs_shifted)), fail_corrs_shifted.values, color=colors, alpha=0.8)
    axes[1, 1].set_yticks(range(len(fail_corrs_shifted)))
    axes[1, 1].set_yticklabels(fail_corrs_shifted.index, fontsize=11)
    axes[1, 1].axvline(x=0, color='black', linewidth=0.8)
    axes[1, 1].set_xlabel('Коэффициент корреляции', fontsize=12)
    axes[1, 1].set_title(f'Влияние погоды на {fail_col} (сдвиг +1 день)', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlim([-1, 1])
    axes[1, 1].tick_params(labelsize=10)
    for i, v in enumerate(fail_corrs_shifted.values):
        axes[1, 1].text(v + (0.05 if v > 0 else -0.15), i, f'{v:.3f}', va='center', fontsize=10)

plt.suptitle('Анализ корреляции со сдвигом по времени', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_with_shift.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nВСЕГО ЗАПИСЕЙ: {len(df_analysis)}")
if corr_shifted is not None:
    print(f"ЗАПИСЕЙ ДЛЯ АНАЛИЗА СО СДВИГОМ: {len(df_shifted)}\n")

if fail_col and len(weather_cols) > 0:
    print(f"КОРРЕЛЯЦИИ С {fail_col} (БЕЗ СДВИГА):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = df_analysis[fail_col].corr(df_analysis[weather_col])
        p_value = stats.pearsonr(df_analysis[fail_col].dropna(), df_analysis[weather_col].dropna())[1]
        print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

if corr_shifted is not None and len(df_shifted) > 0:
    print(f"\nКОРРЕЛЯЦИИ С {fail_col} (ПОГОДА СО СДВИГОМ +1 ДЕНЬ):")
    print("-" * 50)
    for weather_col in weather_cols:
        corr = corr_shifted[fail_col][weather_col]
        if len(df_shifted[weather_col].dropna()) > 2:
            p_value = stats.pearsonr(df_shifted[fail_col].dropna(), df_shifted[weather_col].dropna())[1]
            print(f"{weather_col:15s}: r = {corr:7.3f} (p = {p_value:.4f})")

    print(f"\nСРАВНЕНИЕ КОРРЕЛЯЦИЙ:")
    print("-" * 50)
    for weather_col in weather_cols:
        corr_no_shift = df_analysis[fail_col].corr(df_analysis[weather_col])
        corr_with_shift = corr_shifted[fail_col][weather_col]
        diff = abs(corr_with_shift) - abs(corr_no_shift)
        marker = "↑" if diff > 0.05 else ("↓" if diff < -0.05 else "=")
        print(f"{weather_col:15s}: без сдвига={corr_no_shift:6.3f}, со сдвигом={corr_with_shift:6.3f} {marker}")

- Корреляция < 99.8% (сдвиг погоды внутри вышки)  
Сохранено в `correlation_per_tower_shift.png`
- Динамика корреляции Cell Avail Base Tech (%) (сдвиг внутри вышки)  
Сохранено в `correlation_dynamics_per_tower.png`
- Таблица корреляций для Cell Avail Base Tech (%)
####

In [ ]:
tower_col = None
for col in df.columns:
    if any(x in col.lower() for x in ['tower', 'cell', 'site', 'id', 'вышка', 'базовая']):
        if col not in tech_cols + weather_cols:
            tower_col = col
            break

date_col = None
if 'Date' in df.columns: date_col = 'Date'
elif 'Дата' in df.columns: date_col = 'Дата'

if not tower_col or not date_col:
    print(f"Не найдены нужные колонки. Найдено: tower={tower_col}, date={date_col}")
    print(f"Доступные колонки: {list(df.columns)}")
    exit()

print(f"Используем: вышка='{tower_col}', дата='{date_col}'")

df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values([tower_col, date_col]).reset_index(drop=True)

df_filtered = df[df['Cell Avail Base Tech (%)'] < 99.8].copy()
print(f"Записей с доступностью < 99.8%: {len(df_filtered)}")
print(f"Уникальных вышек: {df_filtered[tower_col].nunique()}")

def get_shifted_weather(df, weather_cols, tower_col, date_col, shift_days):
    """Сдвигает погодные колонки на shift_days дней ВНУТРИ каждой вышки"""
    result = df.copy()
    for col in weather_cols:
        result[col] = df.groupby(tower_col)[col].shift(shift_days)
    return result

shifts = [0, 1, 2, 3, 4, 5, 6, 7]
correlations = {}

for s in shifts:
    if s == 0:
        data = df_filtered[tech_cols + weather_cols].dropna()
    else:
        data = df_filtered[tech_cols + weather_cols + [tower_col, date_col]].copy()
        for col in weather_cols:
            data[col] = df_filtered.groupby(tower_col)[col].shift(s)
        data = data.dropna()

    if len(data) > 10:
        key = 'No Shift' if s == 0 else f'Shift +{s} Day' if s == 1 else f'Shift +{s} Days'
        correlations[key] = data[tech_cols + weather_cols].corr()
        print(f"{key}: {len(data)} записей")

In [ ]:
fail_col = 'Cell Avail Base Tech (%)'

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
plot_labels = [k for k in correlations.keys() if 'No Shift' in k or '+1 Day' in k or '+2 Day' in k or '+7 Days' in k][:4]

for i, label in enumerate(plot_labels):
    corr = correlations[label]

    sns.heatmap(corr.loc[tech_cols, weather_cols], annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, linewidths=0.3, ax=axes[0, i], cbar=False)
    axes[0, i].set_title(label, fontsize=10, fontweight='bold')
    plt.setp(axes[0, i].get_xticklabels(), rotation=45, ha='right', fontsize=8)

    if fail_col in tech_cols:
        fail_corrs = corr[fail_col][weather_cols].sort_values(ascending=False)
        colors = ['red' if x > 0 else 'blue' for x in fail_corrs.values]
        axes[1, i].barh(range(len(fail_corrs)), fail_corrs.values, color=colors, alpha=0.7)
        axes[1, i].set_yticks(range(len(fail_corrs)))
        axes[1, i].set_yticklabels(fail_corrs.index, fontsize=8)
        axes[1, i].axvline(x=0, color='black', linewidth=0.5)
        axes[1, i].set_xlim([-1, 1])
        axes[1, i].set_title(f'{fail_col}', fontsize=9)
        for j, v in enumerate(fail_corrs.values):
            axes[1, i].text(v + (0.05 if v > 0 else -0.15), j, f'{v:.2f}', va='center', fontsize=7)

plt.suptitle(f'Корреляция < 99.8% (сдвиг погоды ВНУТРИ вышки)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_per_tower_shift.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
if fail_col in tech_cols:
    fig, ax = plt.subplots(figsize=(12, 6))
    for wc in weather_cols:
        values = [correlations[label][fail_col][wc] for label in correlations.keys()]
        ax.plot(shifts, values, marker='o', label=wc, linewidth=2)
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_xlabel('Сдвиг (дни)', fontsize=12)
    ax.set_ylabel('Коэффициент корреляции', fontsize=12)
    ax.set_title(f'Динамика корреляции {fail_col} (сдвиг внутри вышки)', fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(shifts)
    plt.tight_layout()
    plt.savefig('correlation_dynamics_per_tower.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if fail_col in tech_cols:
    print(f"\nТаблица корреляций для {fail_col}:")
    header = f"{'Параметр':<15}" + "".join([f"{'D+{s}':>10}" for s in shifts])
    print(header)
    print("-" * (15 + 10 * len(shifts)))
    for wc in weather_cols:
        row = f"{wc:<15}"
        for s in shifts:
            key = 'No Shift' if s == 0 else f'Shift +{s} Day' if s == 1 else f'Shift +{s} Days'
            c = correlations[key][fail_col][wc] if key in correlations else np.nan
            row += f"{c:10.3f}"
        print(row)

In [ ]:
corr_matrix = X_train.corr(numeric_only=True)

print("\n Топ-20 признаков с наибольшей дисперсией:")
feature_vars = X_train.var(numeric_only=True).sort_values(ascending=False).head(20)
print(feature_vars)

top_features = feature_vars.index.tolist()
corr_top = corr_matrix.loc[top_features, top_features]

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, mask=mask, cmap='coolwarm', vmin=-1, vmax=1,
            center=0, annot=True, fmt='.2f', square=True,
            linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Матрица корреляций (Топ-20 признаков по дисперсии)', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

X_train_with_target = X_train.copy()
X_train_with_target['target'] = np.asarray(y_train).flatten()

corr_matrix_full = X_train_with_target.corr(numeric_only=True)

if 'target' not in corr_matrix_full.columns:
    raise ValueError(" 'target' не попал в корреляционную матрицу. Убедись, что y_train содержит 0 и 1 (int/float).")

corr_with_target = corr_matrix_full['target'].drop('target').sort_values(ascending=False)

print("\n Топ-15 признаков по корреляции с target:")
print(corr_with_target.head(15))

# Визуализация корреляции с target
plt.figure(figsize=(10, 8))
top_corr_features = corr_with_target.head(20).index
corr_values = corr_with_target.loc[top_corr_features]

colors = ['red' if x < 0 else 'green' for x in corr_values.values]
plt.barh(range(len(corr_values)), corr_values.values, color=colors, alpha=0.7)
plt.yticks(range(len(corr_values)), corr_values.index)
plt.xlabel('Корреляция с target')
plt.title('Корреляция признаков с целевой переменной', fontsize=14)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Поиск сильно коррелирующих пар
high_corr_pairs = []
cols = corr_matrix.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.8:
            high_corr_pairs.append({
                'Feature_1': cols[i],
                'Feature_2': cols[j],
                'Correlation': corr_val
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print("\n СИЛЬНО КОРРЕЛИРУЮЩИЕ ПАРЫ (|corr| > 0.8):")
    print(high_corr_df.to_string(index=False))
else:
    print("\n Сильно коррелирующих пар не найдено.")

# Корреляция по группам
groups = {
    'Weather': [c for c in weather_cols if c in X_train.columns],
    'Weather_Next': [c for c in weather_next_cols if c in X_train.columns],
    'Cyclical': [c for c in cyclical_cols if c in X_train.columns],
    'Interaction': [c for c in interaction_cols if c in X_train.columns],
    'History': [c for c in history_cols if c in X_train.columns],
    'Tech': [c for c in tech_cols if c in X_train.columns]
}

print("\n Средняя корреляция внутри групп:")
for group_name, features in groups.items():
    if len(features) > 1:
        group_corr = corr_matrix.loc[features, features]
        upper_tri = group_corr.where(np.triu(np.ones(group_corr.shape), k=1).astype(bool))
        mean_corr = upper_tri.stack().mean()
        print(f"{group_name:15s}: {mean_corr:.3f}")

# Heatmap исторических признаков
if len(history_cols) > 1:
    hist_features = [c for c in history_cols if c in X_train.columns]
    if len(hist_features) > 1:
        plt.figure(figsize=(10, 8))
        corr_hist = corr_matrix.loc[hist_features, hist_features]
        mask = np.triu(np.ones_like(corr_hist, dtype=bool))
        sns.heatmap(corr_hist, mask=mask, cmap='YlGnBu', annot=True, fmt='.3f',
                    square=True, linewidths=.5)
        plt.title('Корреляция исторических признаков', fontsize=14, pad=20)
        plt.tight_layout()
        plt.show()

### 04: Baseline


| Модель | Accuracy | ROC-AUC | Precision (Авария) | Recall (Авария) | F1-score (Авария) | Precision (Нет аварии) | Recall (Нет аварии) | F1-score (Нет аварии) | % найденных аварий |
|--------|----------|---------|--------------------|-----------------|-------------------|------------------------|---------------------|-----------------------|---------------------|
| Logistic Regression | 0.8272 | 0.6327 | 0.0753 | 0.3533 | 0.1241 | 0.9732 | 0.8442 | 0.9042 | 35.3% |
| Random Forest | 0.9665 | 0.7128 | 0.1376 | 0.3796 | 0.2020 | 0.9762 | 0.9146 | 0.9444 | 37.9% |
| GaussianNB | 0.9578 | 0.625 | 0.23 | 0.13 | 0.17 | 0.97 | 0.99 | 0.98 | 13.1% |

#### Разделение данных

Случайное разделение

In [ ]:
df['target_today'] = (df['Cell Avail Base Tech (%)'] < 99.8).astype(int)

weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']
feature_cols = [c for c in weather_cols if c in df.columns]

X = df[feature_cols].fillna(0)
y = df['target_today']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Разделение по времени

In [ ]:
df['RECDATE'] = pd.to_datetime(df['RECDATE'])
df = df.sort_values(['Master Site', 'RECDATE']).reset_index(drop=True)

split_date = df['RECDATE'].quantile(0.8)
train_df = df[df['RECDATE'] <= split_date].copy()
test_df  = df[df['RECDATE'] >  split_date].copy()

In [ ]:
import pandas as pd
import numpy as np

train_df = create_features(train_df)
train_df = train_df.loc[:, ~train_df.columns.duplicated()]

BUFFER_DAYS = 30
test_buffer = train_df.groupby('Master Site').tail(BUFFER_DAYS)

test_buffer = test_buffer.loc[:, ~test_buffer.columns.duplicated()]
test_df     = test_df.loc[:, ~test_df.columns.duplicated()]

test_with_hist = pd.concat([test_buffer, test_df], ignore_index=True) \
                   .sort_values(['Master Site', 'RECDATE']) \
                   .reset_index(drop=True)

test_with_hist = create_features(test_with_hist)
test_with_hist = test_with_hist.loc[:, ~test_with_hist.columns.duplicated()]

test_df = test_with_hist[test_with_hist['RECDATE'] > split_date].copy()

train_df['target_today'] = (train_df['Cell Avail Base Tech (%)'] < 99.8).astype(int)
test_df['target_today']  = (test_df['Cell Avail Base Tech (%)'] < 99.8).astype(int)

train_df['target_next_day'] = train_df.groupby('Master Site')['target_today'].shift(-1)
test_df['target_next_day']  = test_df.groupby('Master Site')['target_today'].shift(-1)

train_df = train_df.dropna(subset=['target_next_day'])
test_df  = test_df.dropna(subset=['target_next_day'])

tech_cols = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']
cyclical_cols = ['day_sin', 'day_cos', 'day_month_sin', 'day_month_cos',
                 'month_sin', 'month_cos', 'quarter_sin', 'quarter_cos',
                 'season_sin', 'season_cos']
interaction_cols = ['Storm_Intensity', 'Ice_Storm_Risk', 'Precip_Extremes',
                    'Hail_Impact', 'Winter_Storm', 'Winter_Intense_Precip', 'Summer_Intense_Rain']
history_cols = ['Failures_7d', 'Failures_30d', 'Days_Since_Fail',
                '4G_EMA_3', 'BaseTech_EMA_3', 'Hist_Avg_Avail']

all_feature_names = tech_cols + weather_cols + weather_next_cols + cyclical_cols + \
                    interaction_cols + history_cols + alarm_cols

feature_cols = [c for c in all_feature_names if c in train_df.columns]

X_train = train_df[feature_cols].fillna(0)
y_train = train_df['target_next_day'].astype(int)

X_test  = test_df[feature_cols].fillna(0)
y_test  = test_df['target_next_day'].astype(int)

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Features: {len(feature_cols)}")

In [ ]:
train_df.head()

In [ ]:

X_train_copy = X_train.copy()
X_train_copy['target_today'] = train_df['target_today'].values

corr_today = X_train_copy.corr(numeric_only=True)['target_today'].drop('target_today').sort_values(ascending=False)

print("Корреляция признаков с СЕГОДНЯШНИМ target (проверка на leakage):")
print(corr_today.head(10))


#### LogisticRegression

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, solver='lbfgs')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "="*60)
print("ПРОГНОЗ НА СЛЕДУЮЩИЙ ДЕНЬ (Logistic Regression)")
print("="*60)
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=['OK (>=99.8%)', 'Problem (<99.8%)'], digits=4))

coef_df = pd.DataFrame({'feature': feature_cols, 'coef': model.coef_[0]}).sort_values('coef', key=abs, ascending=False)
print("\nТоп признаков:")
print(coef_df.head(10).to_string(index=False))

In [ ]:
df_model['target_today'] = (df_model['Cell Avail Base Tech (%)'] < 99.8).astype(int)
corr = df_model['target_today'].corr(df_model['target_next_day'])
print(f"Автокорреляция целевой переменной: {corr:.4f}")

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
    solver='lbfgs')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "="*60)
print("ПРОГНОЗ НА СЛЕДУЮЩИЙ ДЕНЬ (Logistic Regression)")
print("="*60)
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=['OK (>=99.8%)', 'Problem (<99.8%)'], digits=4))

coef_df = pd.DataFrame({'feature': feature_cols, 'coef': model.coef_[0]}).sort_values('coef', key=abs, ascending=False)
print("\nТоп признаков:")
print(coef_df.head(10).to_string(index=False))

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred   = rf_model.predict(X_test)

print("\n" + "="*60)
print("ПРОГНОЗ НА СЛЕДУЮЩИЙ ДЕНЬ (Random Forest)")
print("="*60)
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=['OK (>=99.8%)', 'Problem (<99.8%)'],
                            digits=4))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

feature_names = list(X_train.columns)
importances = rf_model.feature_importances_
df_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\n" + "="*60)
print("ВАЖНОСТЬ ПРИЗНАКОВ (Random Forest)")
print("="*60)
print(df_imp.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
top_n = 15
sns.barplot(x='Importance', y='Feature', data=df_imp.head(top_n), palette='viridis')
plt.title(f'Топ {top_n} признаков (Random Forest)', fontsize=14)
plt.xlabel('Gini Importance', fontsize=12)
plt.ylabel('Признак', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

y_pred   = rf_model.predict(X_test)
y_proba  = rf_model.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("ПРОГНОЗ НА СЛЕДУЮЩИЙ ДЕНЬ (Random Forest)")
print("="*60)
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=['OK (>=99.8%)', 'Problem (<99.8%)'],
                            digits=4))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

feature_names = list(X_train.columns)
importances = rf_model.feature_importances_
df_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\n" + "="*60)
print("ВАЖНОСТЬ ПРИЗНАКОВ (Random Forest)")
print("="*60)
print(df_imp.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
top_n = 15
sns.barplot(x='Importance', y='Feature', data=df_imp.head(top_n), palette='viridis')
plt.title(f'Топ {top_n} признаков (Random Forest)', fontsize=14)
plt.xlabel('Gini Importance', fontsize=12)
plt.ylabel('Признак', fontsize=12)
plt.tight_layout()
plt.show()

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
model = GaussianNB()
model.fit(X_train, y_train)

y_pred_default = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_default):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred_default))
print("\nОтчёт по классам:")
print(classification_report(y_test, y_pred_default, target_names=['Нет аварии', 'Авария']))

### 05: Кластеризация

In [ ]:
df = df_main_with_weather.copy()

df_unique = df.drop_duplicates(subset=['Широта WGS84', 'Долгота WGS84'])
df_unique = df_unique.dropna(subset=['Широта WGS84', 'Долгота WGS84'])
df_scaled = df_unique.copy()

#### Scatter plot координаты вышек

In [ ]:
scaler = StandardScaler()
df_scaled[['Широта_scaled', 'Долгота_scaled']] = scaler.fit_transform(df_unique[['Широта WGS84', 'Долгота WGS84']])


xy = np.vstack([df_scaled['Долгота_scaled'], df_scaled['Широта_scaled']])
z = stats.gaussian_kde(xy)(xy)

idx = z.argsort()
x_sorted = df_scaled['Долгота_scaled'].iloc[idx]
y_sorted = df_scaled['Широта_scaled'].iloc[idx]
z_sorted = z[idx]

plt.figure(figsize=(10, 8))
scatter = plt.scatter(x_sorted, y_sorted, c=z_sorted, s=30, alpha=0.8,
                     cmap='hot', edgecolors='black', linewidth=0.3)
plt.colorbar(scatter, label='Плотность точек')
plt.xlabel('Долгота', fontsize=12)
plt.ylabel('Широта', fontsize=12)
plt.title('Scatter plot с градиентом плотности', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### K-means

In [ ]:
lat = df_unique['Широта WGS84'].values
lon = df_unique['Долгота WGS84'].values
lat_rad = np.radians(lat)
lon_rad = np.radians(lon)
lat0 = np.mean(lat_rad)
lon0 = np.mean(lon_rad)
R = 6371
x_km = R * (lon_rad - lon0) * np.cos(lat0)
y_km = R * (lat_rad - lat0)
X_km = np.column_stack((x_km, y_km))

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df_scaled['KMeans_Cluster'] = kmeans.fit_predict(X_km)

kmeans_silhouette = silhouette_score(X_km, df_scaled['KMeans_Cluster'])
print(f"Silhouette Score финальной модели: {kmeans_silhouette:.4f}")


plt.figure(figsize=(10, 8))
plt.scatter(X_km[:, 0], X_km[:, 1], c=df_scaled['KMeans_Cluster'], cmap='tab10', s=30, alpha=0.6)

#### DBSCAN

In [ ]:
dbscan = DBSCAN(eps=4, min_samples=5)
labels = dbscan.fit_predict(X_km)

df_scaled['DBSCAN_Cluster'] = labels
mask = labels != -1
score = silhouette_score(X_km[mask], labels[mask])
print(f"Silhouette Score: {score:.4f}")

plt.figure(figsize=(10, 8))
plt.scatter(X_km[:, 0], X_km[:, 1], c=labels, cmap='tab10', s=30, alpha=0.6)

In [ ]:
from sklearn.cluster import KMeans, DBSCAN

def calculate_tower_metrics(y_true, y_pred, y_prob, tower_ids):
    metrics = []
    for tower_id in tower_ids.unique():
        mask = tower_ids == tower_id
        y_t = y_true[mask].values
        y_p = y_pred[mask]
        y_pr = y_prob[mask]

        if len(y_t) < 10:
            continue

        if len(np.unique(y_t)) == 2:
            auc = roc_auc_score(y_t, y_pr)
            pr_auc = average_precision_score(y_t, y_pr)
        else:
            auc = np.nan
            pr_auc = np.nan

        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        metrics.append({
            'Master_Site': tower_id,
            'auc_roc': auc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })
    return pd.DataFrame(metrics)

tower_metrics = calculate_tower_metrics(y_test, y_pred_lgb, y_prob_lgb, test_df['Master Site'])

feature_cols = ['auc_roc', 'precision', 'recall', 'f1_score']
X_cluster = tower_metrics[feature_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("K-MEANS КЛАСТЕРИЗАЦИЯ")
print("="*60)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
tower_metrics.loc[X_cluster.index, 'kmeans_cluster'] = kmeans.fit_predict(X_scaled)

for cluster_id in range(n_clusters):
    cluster_data = tower_metrics[tower_metrics['kmeans_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")
    print(f"  Recall:   {cluster_data['recall'].mean():.3f}")

print("\n" + "="*60)
print("DBSCAN КЛАСТЕРИЗАЦИЯ")
print("="*60)

dbscan = DBSCAN(eps=0.8, min_samples=3)
tower_metrics.loc[X_cluster.index, 'dbscan_cluster'] = dbscan.fit_predict(X_scaled)

n_dbscan_clusters = len(set(tower_metrics['dbscan_cluster'])) - (1 if -1 in tower_metrics['dbscan_cluster'].values else 0)
n_noise = list(tower_metrics['dbscan_cluster']).count(-1)

print(f"Найдено кластеров: {n_dbscan_clusters}")
print(f"Шумовых точек: {n_noise}")

for cluster_id in sorted(set(tower_metrics['dbscan_cluster'])):
    if cluster_id == -1:
        continue
    cluster_data = tower_metrics[tower_metrics['dbscan_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")



In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

def calculate_tower_metrics(y_true, y_pred, y_prob, tower_ids):
    metrics = []
    for tower_id in tower_ids.unique():
        mask = tower_ids == tower_id
        y_t = y_true[mask].values
        y_p = y_pred[mask]
        y_pr = y_prob[mask]

        if len(y_t) < 10:
            continue

        if len(np.unique(y_t)) == 2:
            auc = roc_auc_score(y_t, y_pr)
            pr_auc = average_precision_score(y_t, y_pr)
        else:
            auc = np.nan
            pr_auc = np.nan

        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        metrics.append({
            'Master_Site': tower_id,
            'auc_roc': auc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })
    return pd.DataFrame(metrics)

tower_metrics = calculate_tower_metrics(y_test, y_pred_lgb, y_prob_lgb, test_df['Master Site'])

feature_cols = ['auc_roc', 'precision', 'recall', 'f1_score']
X_cluster = tower_metrics[feature_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("K-MEANS КЛАСТЕРИЗАЦИЯ")
print("="*60)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
tower_metrics.loc[X_cluster.index, 'kmeans_cluster'] = kmeans.fit_predict(X_scaled)

for cluster_id in range(n_clusters):
    cluster_data = tower_metrics[tower_metrics['kmeans_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")
    print(f"  Recall:   {cluster_data['recall'].mean():.3f}")

print("\n" + "="*60)
print("DBSCAN КЛАСТЕРИЗАЦИЯ")
print("="*60)

dbscan = DBSCAN(eps=0.8, min_samples=3)
tower_metrics.loc[X_cluster.index, 'dbscan_cluster'] = dbscan.fit_predict(X_scaled)

n_dbscan_clusters = len(set(tower_metrics['dbscan_cluster'])) - (1 if -1 in tower_metrics['dbscan_cluster'].values else 0)
n_noise = list(tower_metrics['dbscan_cluster']).count(-1)

print(f"Найдено кластеров: {n_dbscan_clusters}")
print(f"Шумовых точек: {n_noise}")

for cluster_id in sorted(set(tower_metrics['dbscan_cluster'])):
    if cluster_id == -1:
        continue
    cluster_data = tower_metrics[tower_metrics['dbscan_cluster'] == cluster_id]
    print(f"\nКластер {cluster_id}: {len(cluster_data)} вышек")
    print(f"  AUC-ROC:  {cluster_data['auc_roc'].mean():.3f}")
    print(f"  F1-Score: {cluster_data['f1_score'].mean():.3f}")

print("\n" + "="*60)
print("СРАВНЕНИЕ С ГЕОГРАФИЧЕСКОЙ КЛАСТЕРИЗАЦИЕЙ")
print("="*60)

geo_col = 'Subregion' 

if geo_col in test_df.columns:
    tower_geo = test_df[['Master Site', geo_col]].drop_duplicates()
    merged = tower_metrics.merge(tower_geo, on='Master_Site', how='left')

    print("\nK-Means × География:")
    print(pd.crosstab(merged['kmeans_cluster'], merged[geo_col]))

    print("\nDBSCAN × География:")
    print(pd.crosstab(merged['dbscan_cluster'], merged[geo_col]))
else:
    print(f"Колонка {geo_col} не найдена!")

tower_metrics.to_csv('tower_clusters.csv', index=False)
print("\nРезультаты сохранены в tower_clusters.csv")

### 06: Признаки

#### Сезонность

In [ ]:
df = df_main_with_weather.copy()

In [ ]:
df['RECDATE'] = pd.to_datetime(df['RECDATE'])

# День недели (0=Пн, 6=Вс)
df['day_of_week'] = df['RECDATE'].dt.dayofweek
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# День месяца (1-31)
df['day_of_month'] = df['RECDATE'].dt.day
df['day_month_sin'] = np.sin(2 * np.pi * df['day_of_month'] / 31)
df['day_month_cos'] = np.cos(2 * np.pi * df['day_of_month'] / 31)

# Месяц (1-12)
df['month'] = df['RECDATE'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Квартал (1-4)
df['quarter'] = df['RECDATE'].dt.quarter
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

# Сезон
df['season'] = df['month'].map({
    12: 'winter', 1: 'winter', 2: 'winter',
    3: 'spring', 4: 'spring', 5: 'spring',
    6: 'summer', 7: 'summer', 8: 'summer',
    9: 'autumn', 10: 'autumn', 11: 'autumn'
})

# Циклическое кодирование сезона
season_num_map = {'winter': 0, 'spring': 1, 'summer': 2, 'autumn': 3}
df['season_num'] = df['season'].map(season_num_map)

df['season_sin'] = np.sin(2 * np.pi * df['season_num'] / 4)
df['season_cos'] = np.cos(2 * np.pi * df['season_num'] / 4)

df.drop(columns=['season_num'], inplace=True)

weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']
weather_next_cols = []
for col in weather_cols:
    col_next = f'{col}_next'
    df[col_next] = df.groupby('Master Site')[col].shift(-1)
    weather_next_cols.append(col_next)

#### Составные погодные признаки

In [ ]:
# 1. Штормовой индекс
df['Storm_Intensity'] = df['MaxWind'] * df['MaxInt']

# 2. Риск обледенения
df['Ice_Storm_Risk'] = df['TempDelta'] * df['MaxInt'] * (df['Snow'] + df['Rain'] + df['Sprinkling'])

# 3. Экстремальные осадки
df['Precip_Extremes'] = df['MaxInt'] * (df['Rain'] + df['Snow'] + df['Hail'])

# 4. Град + интенсивность
df['Hail_Impact'] = df['Hail'] * df['MaxInt']

# 5. Зимний шторм
df['Winter_Storm'] = df['Snow'] * df['MaxInt'] * df['MaxWind']

# 7. Сезон × Интенсивность
df['Winter_Intense_Precip'] = df['MaxInt'] * (df['season'] == 'winter').astype(int)
df['Summer_Intense_Rain'] = df['MaxInt'] * df['Rain'] * (df['season'] == 'summer').astype(int)

#### Исторические признаки

Запускать на разделенном на тест и трейн датасете как функцию

In [ ]:
def create_features(data):
    data = data.copy()

    data['is_failure'] = (data['Cell Avail Base Tech (%)'] < 99.8).astype(int)

    data['Failures_7d']  = data.groupby('Master Site')['is_failure'].transform(lambda x: x.rolling(7, min_periods=1).sum())
    data['Failures_30d'] = data.groupby('Master Site')['is_failure'].transform(lambda x: x.rolling(30, min_periods=1).sum())

    data['Last_Fail_Date'] = data['RECDATE'].where(data['is_failure'] == 1, pd.NaT)
    data['Last_Fail_Date'] = data.groupby('Master Site')['Last_Fail_Date'].ffill()
    data['Days_Since_Fail'] = (data['RECDATE'] - data['Last_Fail_Date']).dt.days
    data['Days_Since_Fail'] = data['Days_Since_Fail'].fillna(999).astype(int)

    data['4G_EMA_3']       = data.groupby('Master Site')['Cell Avail 4G (%)'].transform(lambda x: x.ewm(span=3, adjust=False).mean())
    data['BaseTech_EMA_3'] = data.groupby('Master Site')['Cell Avail Base Tech (%)'].transform(lambda x: x.ewm(span=3, adjust=False).mean())

    data['Hist_Avg_Avail'] = data.groupby('Master Site')['Cell Avail Base Tech (%)'].transform(lambda x: x.expanding().mean())

    data.drop(columns=['is_failure', 'Last_Fail_Date'], inplace=True, errors='ignore')

    return data

# Пример:
# train_df = create_features(train_df)
# test_df  = create_features(test_df)


#### Причины аварий

In [ ]:
import re

df['Alarm_Clean'] = df['Alarm Descriptions'].fillna('').str.strip()
df['Alarm_Clean'] = df['Alarm_Clean'].replace('', 'No_alarms')
df['Alarm_Clean'] = df['Alarm_Clean'].str.replace(r'\s*[|;,]\s*', '|', regex=True)

alarm_dummies = df['Alarm_Clean'].str.get_dummies(sep='|')

clean_names = []
for col in alarm_dummies.columns:
    safe_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
    safe_name = re.sub(r'_+', '_', safe_name).strip('_')
    clean_names.append(f'Alarm_{safe_name}')

alarm_dummies.columns = clean_names

if 'Alarm_No_alarms' in alarm_dummies.columns:
    alarm_dummies = alarm_dummies.drop(columns=['Alarm_No_alarms'])

df = pd.concat([df, alarm_dummies], axis=1)
df.drop(columns=['Alarm_Clean'], inplace=True)

alarm_cols = [c for c in df.columns if c.startswith('Alarm_')]

### 07: Бустинг

| Модель      | Accuracy | ROC-AUC | Precision (Авария) | Recall (Авария) | F1-score (Авария) | Precision (Нет аварии) | Recall (Нет аварии) | F1-score (Нет аварии) | % найденных аварий |
|-------------|----------|---------|--------------------|-----------------|-------------------|------------------------|---------------------|-----------------------|---------------------|
| XGBoost     | 0.78     | 0.75       | 0.08               | 0.58            | 0.14              | 0.98                   | 0.79                | 0.87                  | 58.0%               |
| LightGBM    | 0.77     | 0.75       | 0.08               | 0.59            | 0.14              | 0.98                   | 0.77                | 0.87                  | 59.0%               |
| CatBoost    | 0.77     | —       | 0.08               | 0.59            | 0.14              | 0.98                   | 0.78                | 0.87                  | 59.0%               |

После оптуны (максимизировали average_precision_score):

| Модель      | Accuracy | ROC-AUC | Precision (Авария) | Recall (Авария) | F1-score (Авария) | Precision (Нет аварии) | Recall (Нет аварии) | F1-score (Нет аварии) | % найденных аварий |
|-------------|----------|---------|--------------------|-----------------|-------------------|------------------------|---------------------|-----------------------|---------------------|
| XGBoost     | 0.86     | 0.73       | 0.11               | 0.44            | 0.17              | 0.98                   | 0.88                | 0.93                  | 44.0%               |
| LightGBM    | 0.80     | 0.74       | 0.09               | 0.53            | 0.15              | 0.98                   | 0.81                | 0.89                  | 53.0%               |
| CatBoost    | 0.85     | 0.73       | 0.10               | 0.48            | 0.17              | 0.98                   | 0.86                | 0.92                  | 48.0%               |

roc_auc_score:

| Модель    | Accuracy | ROC-AUC | Precision (Авария) | Recall (Авария) | F1-score (Авария) | Precision (Нет аварии) | Recall (Нет аварии) | F1-score (Нет аварии) | % найденных аварий |
|-----------|----------|---------|--------------------|------------------|----------------------|------------------------|----------------------|------------------------|----------------------|
| XGBoost   | 0.82     | 0.746   | 0.09               | 0.51             | 0.16                 | 0.98                   | 0.83                 | 0.90                   | 51.0%                |
| LightGBM  | 0.78     | 0.748   | 0.08               | 0.58             | 0.14                 | 0.98                   | 0.78                 | 0.87                   | 58.0%                |
| CatBoost  | 0.78     | 0.749   | 0.08               | 0.58             | 0.14                 | 0.98                   | 0.78                 | 0.87                   | 58.0%                |

после кросс-валидации:

| Модель    | Accuracy | ROC-AUC | Precision (Авария) | Recall (Авария) | F1-score (Авария) | Precision (Нет аварии) | Recall (Нет аварии) | F1-score (Нет аварии) | % найденных аварий |
|-----------|----------|---------|--------------------|------------------|----------------------|------------------------|----------------------|------------------------|----------------------|
| XGBoost   | 0.78     | 0.749   | 0.08               | 0.58             | 0.14                 | 0.98                   | 0.79                 | 0.87                   | 58.0%                |
| LightGBM  | 0.76     | 0.750   | 0.08               | 0.60             | 0.14                 | 0.98                   | 0.77                 | 0.86                   | 60.0%                |
| CatBoost  | 0.78     | 0.748   | 0.08               | 0.58             | 0.14                 | 0.98                   | 0.79                 | 0.87                   | 58.0%                |

До запуска ячейки использовать 06 для признаков (и в т.ч. для импорта)

In [ ]:
!pip install catboost

In [ ]:
!pip install optuna

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna

#### XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

def objective(trial):
    def get_model():
        return xgb.XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 30, 2000),
            max_depth=trial.suggest_int("max_depth", 1, 13),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            subsample=trial.suggest_float("subsample", 0.1, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            gamma=trial.suggest_float("gamma", 0, 5),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric="logloss",
            verbose=0
        )

    cv_res = run_time_series_cv(X_train, y_train, get_model, n_splits=3, use_scaler=False, verbose=False)
    return cv_res['mean_auc']

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200)
best_params = study.best_params

model_xgb = xgb.XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    verbose=0
)

model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
y_prob_xgb = model_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print("roc-auc ", roc_auc_score(y_test, y_prob_xgb))
print("pr-auc ", average_precision_score(y_test, y_prob_xgb))

print(f"Best params: {best_params}")
print(f"Best AUC: {study.best_value}")

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_xgb = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    verbose=0
)

model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
y_prob_xgb = model_xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))
print("PR-AUC: ", average_precision_score(y_test, y_prob_xgb))

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, thresholds = precision_recall_curve(y_test, y_prob_xgb)
pr_auc = average_precision_score(y_test, y_prob_xgb)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()

plt.grid(True)
plt.show()

In [ ]:
feat_imp_xgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_xgb.feature_importances_
}).sort_values('importance', ascending=False)
print(feat_imp_xgb.head(10))


In [ ]:
hist_features = ['Failures_7d', 'Failures_30d', 'Days_Since_Fail',
                 '4G_EMA_3', 'BaseTech_EMA_3', 'Hist_Avg_Avail']
hist_to_drop = [f for f in hist_features if f in X_train.columns]

print(f"\n ABLATION STUDY: удаляем {len(hist_to_drop)} исторических признаков:")
print(f"   {hist_to_drop}")

X_train_abl = X_train.drop(columns=hist_to_drop)
X_test_abl  = X_test.drop(columns=hist_to_drop)

model_xgb_abl = xgb.XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    verbose=0
)
model_xgb_abl.fit(X_train_abl, y_train)

y_pred_abl = model_xgb_abl.predict(X_test_abl)
y_prob_abl = model_xgb_abl.predict_proba(X_test_abl)[:, 1]

auc_full = roc_auc_score(y_test, y_prob_xgb)
auc_abl  = roc_auc_score(y_test, y_prob_abl)
pr_full  = average_precision_score(y_test, y_prob_xgb)
pr_abl   = average_precision_score(y_test, y_prob_abl)

print("\n" + "="*50)
print(" MODEL WITHOUT HISTORY (Ablation)")
print("="*50)
print(classification_report(y_test, y_pred_abl, digits=4))
print(f"roc-auc: {auc_abl:.4f}")
print(f"pr-auc:  {pr_abl:.4f}")


In [ ]:
y_pred = (model_xgb.predict_proba(X_test)[:, 1] >= 0.5).astype(int)

mask_TP = (y_test == 1) & (y_pred == 1)  # Пойманные аварии
mask_FN = (y_test == 1) & (y_pred == 0)  # Пропущенные аварии

print(f"Поймано аварий (TP): {mask_TP.sum()}")
print(f"Пропущено аварий (FN): {mask_FN.sum()}")

print("\n Средние значения исторических признаков:")
comparison = pd.DataFrame({
    'TP (пойманные)': X_test.loc[mask_TP, hist_features].mean(),
    'FN (пропущенные)': X_test.loc[mask_FN, hist_features].mean(),
    'Разница': X_test.loc[mask_TP, hist_features].mean() - X_test.loc[mask_FN, hist_features].mean()
})
print(comparison)

# Визуализация
plt.figure(figsize=(10, 6))
comparison['TP (пойманные)'].plot(kind='bar', color='green', alpha=0.6, label='TP')
comparison['FN (пропущенные)'].plot(kind='bar', alpha=0.6, label='FN')
plt.title('Пойманные vs Пропущенные аварии: средние значения признаков')
plt.ylabel('Значение признака')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### LightGBM

In [ ]:
# Убираем дублирующиеся колонки
print(f"Колонок до очистки: {X_train.shape[1]}")
X_train = X_train.loc[:, ~X_train.columns.duplicated()]
X_test  = X_test.loc[:, ~X_test.columns.duplicated()]
print(f"Колонок после очистки: {X_train.shape[1]}")

# Проверяем, что нет дублей
assert X_train.columns.is_unique, "Есть дубли в X_train!"
assert X_test.columns.is_unique, "Есть дубли в X_test!"


In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

def objective(trial):
    def get_model():
        return lgb.LGBMClassifier(
            n_estimators=trial.suggest_int("n_estimators", 30, 2000),
            max_depth=trial.suggest_int("max_depth", 1, 13),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            subsample=trial.suggest_float("subsample", 0.1, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            verbose=-1
        )

    cv_res = run_time_series_cv(X_train, y_train, get_model, n_splits=3, verbose=False)
    return cv_res['mean_auc']

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

best_params = study.best_params

model_lgb = lgb.LGBMClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)

model_lgb.fit(X_train, y_train)

y_pred_lgb = model_lgb.predict(X_test)
y_prob_lgb = model_lgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lgb))
print(confusion_matrix(y_test, y_pred_lgb))
print("roc-auc ", roc_auc_score(y_test, y_prob_lgb))
print("pr-auc ", average_precision_score(y_test, y_prob_lgb))

study.trials_dataframe().to_csv("optuna_results_lgb.csv")
print(f"Best params: {best_params}")
print(f"Best AUC: {study.best_value}")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_lgb = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)

model_lgb.fit(X_train, y_train)

y_pred_lgb = model_lgb.predict(X_test)
y_prob_lgb = model_lgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lgb))
print("PR-AUC: ", average_precision_score(y_test, y_prob_lgb))

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_lgb)
pr_auc = average_precision_score(y_test, y_prob_lgb)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()

plt.grid(True)
plt.show()

In [ ]:
feat_imp_lgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_lgb.feature_importances_
}).sort_values('importance', ascending=False)
print(feat_imp_lgb.head(10))


#### CatBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

def objective(trial):
    def get_model():
        return CatBoostClassifier(
            iterations=trial.suggest_int("iterations", 100, 2000),
            depth=trial.suggest_int("depth", 1, 12),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1e-3, 10, log=True),
            bagging_temperature=trial.suggest_float("bagging_temperature", 0, 1),
            random_strength=trial.suggest_float("random_strength", 1e-3, 10, log=True),
            scale_pos_weight=scale_pos_weight,
            random_seed=42,
            verbose=0
        )

    cv_res = run_time_series_cv(X_train, y_train, get_model, n_splits=3, verbose=False)
    return cv_res['mean_auc']

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200)

best_params = study.best_params

model_cb = CatBoostClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    random_seed=42,
    verbose=0
)

model_cb.fit(X_train, y_train)

y_pred_cb = model_cb.predict(X_test)
y_prob_cb = model_cb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_cb))
print(confusion_matrix(y_test, y_pred_cb))
print("roc-auc ", roc_auc_score(y_test, y_prob_cb))
print("pr-auc ", average_precision_score(y_test, y_prob_cb))

study.trials_dataframe().to_csv("optuna_results_catboost.csv")
print(f"Best params: {best_params}")
print(f"Best AUC: {study.best_value}")

In [ ]:
import catboost as cb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_cb = cb.CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    scale_pos_weight=scale_pos_weight,
    random_seed=42,
    verbose=0
)

model_cb.fit(X_train, y_train)

y_pred_cb = model_cb.predict(X_test)
y_prob_cb = model_cb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_cb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_cb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_cb))
print("PR-AUC: ", average_precision_score(y_test, y_prob_cb))

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_cb)
pr_auc = average_precision_score(y_test, y_prob_cb)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()

plt.grid(True)
plt.show()

In [ ]:
feat_imp_cb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_cb.feature_importances_
}).sort_values('importance', ascending=False)
print(feat_imp_cb.head(10))

### 08: Нейросети

| Модель    | Accuracy | ROC-AUC | Precision (Авария) | Recall (Авария) | F1-score (Авария) | Precision (Нет аварии) | Recall (Нет аварии) | F1-score (Нет аварии) | % найденных аварий |
|-----------|----------|---------|--------------------|------------------|----------------------|------------------------|----------------------|------------------------|----------------------|
| LSTM   | 0.67     | 0.716   | 0.06               | 0.64             | 0.11                 | 0.98                   | 0.67                 | 0.80                   | 64.0%                |
| GRU  | 0.69     | 0.717   | 0.06               | 0.62             | 0.11                 | 0.98                   | 0.69                 | 0.81                   | 62.0%                |

#### LSTM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super(DiceLoss, self).__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        targets = targets.view(-1, 1)
        probs = probs.view(-1, 1)

        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.eps) / (probs.sum() + targets.sum() + self.eps)

        return 1 - dice

X_train_lstm = X_train.values.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm  = X_test.values.reshape(X_test.shape[0], 1, X_test.shape[1])

y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
y_test_t  = torch.tensor(y_test.values, dtype=torch.float32)

X_train_t = torch.tensor(X_train_lstm, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_lstm, dtype=torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t, y_train_t = X_train_t.to(device), y_train_t.to(device)
X_test_t, y_test_t = X_test_t.to(device), y_test_t.to(device)

model = LSTMModel(
    input_size=X_train_lstm.shape[2],
    hidden_size=64,
    num_layers=2
).to(device)

pos_weight = torch.tensor([scale_pos_weight], dtype=torch.float32).to(device)

bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
dice_loss = DiceLoss()

def criterion(logits, targets):
    return bce_loss(logits, targets) + dice_loss(logits, targets)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
model.train()

epochs = 50

for epoch in range(epochs):
    optimizer.zero_grad()

    logits = model(X_train_t).squeeze()
    loss = criterion(logits, y_train_t)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

In [ ]:
baseline_prob = y_train_t.mean().item()
baseline_pred = (torch.full_like(y_test_t, baseline_prob) >= 0.5).int().cpu().numpy()
print("baseline")
print(classification_report(y_test, baseline_pred))

model.eval()

with torch.no_grad():
    logits = model(X_test_t).squeeze()

    y_prob_lstm = torch.sigmoid(logits).cpu().numpy()
    y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

precision, recall, thresholds = precision_recall_curve(y_test, y_prob_lstm)

pr_auc = average_precision_score(y_test, y_prob_lstm)

print("lstm")
print(classification_report(y_test, y_pred_lstm))
print(confusion_matrix(y_test, y_pred_lstm))
print("roc-auc ", roc_auc_score(y_test, y_prob_lstm))
print("pr-auc ", pr_auc)

#### GRU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super(DiceLoss, self).__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = targets.view(-1, 1)
        probs = probs.view(-1, 1)
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.eps) / (probs.sum() + targets.sum() + self.eps)
        return 1 - dice

X_train_gru = X_train.values.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_gru  = X_test.values.reshape(X_test.shape[0], 1, X_test.shape[1])

y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
y_test_t  = torch.tensor(y_test.values, dtype=torch.float32)
X_train_t = torch.tensor(X_train_gru, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_gru, dtype=torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_t, y_train_t = X_train_t.to(device), y_train_t.to(device)
X_test_t, y_test_t = X_test_t.to(device), y_test_t.to(device)

model = GRUModel(
    input_size=X_train_gru.shape[2],
    hidden_size=64,
    num_layers=2
).to(device)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

pos_weight = torch.tensor([scale_pos_weight], dtype=torch.float32).to(device)
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
dice_loss = DiceLoss()

def criterion(logits, targets):
    return bce_loss(logits, targets) + dice_loss(logits, targets)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
model.train()
epochs = 50

for epoch in range(epochs):
    optimizer.zero_grad()

    logits = model(X_train_t).squeeze()
    loss = criterion(logits, y_train_t)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

In [ ]:
baseline_prob = y_train_t.mean().item()
baseline_pred = (torch.full_like(y_test_t, baseline_prob) >= 0.5).int().cpu().numpy()
print("baseline")
print(classification_report(y_test, baseline_pred))

model.eval()
with torch.no_grad():
    logits = model(X_test_t).squeeze()
    y_prob_gru = torch.sigmoid(logits).cpu().numpy()
    y_pred_gru = (y_prob_gru >= 0.5).astype(int)

pr_auc = average_precision_score(y_test, y_prob_gru)

print("\ngru")
print(classification_report(y_test, y_pred_gru))
print(confusion_matrix(y_test, y_pred_gru))
print("roc-auc ", roc_auc_score(y_test, y_prob_gru))
print("pr-auc ", pr_auc)

### 09: Кросс-валидация

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def run_time_series_cv(X, y, model_factory, n_splits=5, use_scaler=True, verbose=True):
    """
    Запускает TimeSeriesCV для любого классификатора.

    Parameters:
    -----------
    X, y : pandas DataFrame/Series
    model_factory : функция, возвращающая НОВЫЙ экземпляр модели при каждом вызове
    n_splits : int, количество фолдов
    use_scaler : bool, применять StandardScaler внутри каждого фолда
    verbose : bool, выводить логи

    Returns:
    --------
    dict с ключами: scores, models, oof_proba, mean_auc, std_auc
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    cv_scores = []
    cv_models = []
    oof_proba = np.zeros(len(X))

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if use_scaler:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_val = scaler.transform(X_val)

        model = model_factory()
        model.fit(X_tr, y_tr)

        y_proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_proba)
        cv_scores.append(score)
        cv_models.append(model)
        oof_proba[val_idx] = y_proba

        if verbose:
            print(f"Fold {fold} | ROC-AUC: {score:.4f} | Train: {len(X_tr)} | Val: {len(X_val)}")

    mean_auc = np.mean(cv_scores)
    std_auc = np.std(cv_scores)
    if verbose:
        print(f"\n CV Result: {mean_auc:.4f} ± {std_auc:.4f}")

    return {
        "scores": cv_scores,
        "models": cv_models,
        "oof_proba": oof_proba,
        "mean_auc": mean_auc,
        "std_auc": std_auc
    }

###10: Анонимизация данных

In [ ]:
import hashlib

OUTPUT_FILE = 'stations_anonymized.xlsx'

ID_COL      = 'Master Site'
DUP_ID_COL  = 'Номер сайта'
ADDR_COL    = 'Адрес'
LAT_COL     = 'Широта WGS84'
LON_COL     = 'Долгота WGS84'

df = df_main_with_weather.copy()

df.columns = df.columns.str.strip()

cols_to_drop = [col for col in [DUP_ID_COL, ADDR_COL] if col in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f" Удалены колонки: {cols_to_drop}")

SALT = "geo_project_2024"
def hash_station(code):
    if pd.isna(code):
        return np.nan
    return hashlib.sha256(f"{SALT}_{str(code).strip()}".encode()).hexdigest()[:12]

df[ID_COL] = df[ID_COL].apply(hash_station)
print(" Коды станций захэшированы")

valid_mask = df[LAT_COL].notna() & df[LON_COL].notna()
if valid_mask.any():
    coords = df.loc[valid_mask, [LON_COL, LAT_COL]].values.astype(float)

    SCALE      = 1.0          # Масштаб (1.0 = без изменения расстояний)
    ROTATION   = 37           # Угол поворота карты в градусах
    SHIFT      = (59.6, 30.2) # Сдвиг центра (примерно центр СПб/Ленобласти)
    NOISE_STD  = 0.004        # Гауссов шум (400 метров) для приватности

    theta = np.radians(ROTATION)
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta), np.cos(theta)]])

    synth_coords = (coords * SCALE) @ R.T + np.array(SHIFT)
    synth_coords += np.random.default_rng(42).normal(0, NOISE_STD, synth_coords.shape)

    df.loc[valid_mask, LON_COL] = synth_coords[:, 0]
    df.loc[valid_mask, LAT_COL] = synth_coords[:, 1]
    print(" Координаты заменены на синтетические (геометрия сохранена)")

df.to_excel(OUTPUT_FILE, index=False)
print(f"\n Файл сохранён как: {OUTPUT_FILE}")

In [ ]:
df[LAT_COL].head()

### 11: отбор признаков


In [ ]:
from sklearn.feature_selection import SelectFromModel, SelectKBest, mutual_info_classif

In [ ]:
df = df_main_with_weather.copy()
df = df.loc[:, ~df.columns.duplicated()]

df['RECDATE'] = pd.to_datetime(df['RECDATE'])
df = df.sort_values(['Master Site', 'RECDATE']).reset_index(drop=True)
df['target_today'] = (df['Cell Avail Base Tech (%)'] < 99.8).astype(int)
df['target_next_day'] = df.groupby('Master Site')['target_today'].shift(-1)
df = df.dropna(subset=['target_next_day'])
df['target_next_day'] = df['target_next_day'].astype(int)

tech_cols = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']
all_feature_names = tech_cols + weather_cols
feature_cols = [c for c in all_feature_names if c in df.columns]

split_date = df['RECDATE'].quantile(0.8)
train_df = df[df['RECDATE'] <= split_date].copy()
test_df  = df[df['RECDATE'] >  split_date].copy()

X_train = train_df[feature_cols].fillna(0)
y_train = train_df['target_next_day']
X_test  = test_df[feature_cols].fillna(0)
y_test  = test_df['target_next_day']

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
lasso_selector = SelectFromModel(LogisticRegression(
    penalty='l1',
    solver='liblinear',
    random_state=42,
    class_weight='balanced'
))
lasso_selector.fit(X_train_scaled, y_train)

selected_features_lr = X_train_scaled.columns[lasso_selector.get_support()].tolist()
print(f"Отобрано признаков ({len(selected_features_lr)} из {len(feature_cols)}): \n{selected_features_lr}")

X_train_lr = X_train_scaled[selected_features_lr]
X_test_lr = X_test_scaled[selected_features_lr]

# Обучение финальной модели на отобранных признаках
model_lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
    solver='lbfgs'
)
model_lr.fit(X_train_lr, y_train)

y_pred_lr = model_lr.predict(X_test_lr)
y_proba_lr = model_lr.predict_proba(X_test_lr)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=['OK (>=99.8%)', 'Problem (<99.8%)'], digits=4))

coef_df = pd.DataFrame({'feature': selected_features_lr, 'coef': model_lr.coef_[0]}).sort_values('coef', key=abs, ascending=False)
print("Топ признаков (по весам в финальной модели):")
print(coef_df.head(10).to_string(index=False))

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'
)
rf_base.fit(X_train, y_train)

# Отбираем признаки (чья важность выше средней)
rf_selector = SelectFromModel(rf_base, prefit=True)
selected_features_rf = X_train.columns[rf_selector.get_support()].tolist()
print(f"Отобрано признаков ({len(selected_features_rf)} из {len(feature_cols)}): \n{selected_features_rf}")

X_train_rf = X_train[selected_features_rf]
X_test_rf = X_test[selected_features_rf]

rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'
)
rf_model.fit(X_train_rf, y_train)

y_pred_rf   = rf_model.predict(X_test_rf)
y_proba_rf  = rf_model.predict_proba(X_test_rf)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['OK (>=99.8%)', 'Problem (<99.8%)'], digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# Важность для отобранных признаков
df_imp = pd.DataFrame({
    'Feature': selected_features_rf,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(df_imp.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
top_n = min(15, len(selected_features_rf))
sns.barplot(x='Importance', y='Feature', data=df_imp.head(top_n), palette='viridis')
plt.title(f'Топ {top_n} признаков (Random Forest с отбором)', fontsize=14)
plt.xlabel('Gini Importance', fontsize=12)
plt.ylabel('Признак', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.naive_bayes import GaussianNB

k_best = min(10, len(feature_cols))
mi_selector = SelectKBest(score_func=mutual_info_classif, k=k_best)
mi_selector.fit(X_train, y_train)

selected_features_nb = X_train.columns[mi_selector.get_support()].tolist()
print(f"Отобрано {k_best} лучших признаков по взиамной информации: \n{selected_features_nb}")

X_train_nb = X_train[selected_features_nb]
X_test_nb = X_test[selected_features_nb]

model_nb = GaussianNB()
model_nb.fit(X_train_nb, y_train)

y_pred_nb = model_nb.predict(X_test_nb)
y_proba_nb = model_nb.predict_proba(X_test_nb)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_nb):.4f}")
print("Матрица ошибок:")
print(confusion_matrix(y_test, y_pred_nb))
print("Отчёт по классам:")
print(classification_report(y_test, y_pred_nb, target_names=['Нет аварии', 'Авария']))